Perform one-hot encoding for categorical variables while handling high-cardinality categories through target encoding, frequency-based binning, and dimensionality reduction techniques to prevent feature explosion and maintain interpretability.

In [1]:
# Load final Task 19 dataset
import pandas as pd
import numpy as np

df = pd.read_csv("task19_final_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

df.head()

Dataset loaded successfully!
Dataset Shape: (23483, 23)

Columns:
['product_id', 'category', 'subcategory1', 'subcategory2', 'location', 'channel', 'customer_age', 'review_title', 'review_text', 'rating', 'recommend_flag', 'vader_negative', 'vader_neutral', 'vader_positive', 'vader_compound', 'vader_sentiment_score', 'vader_sentiment_label', 'age_cluster_raw', 'age_group', 'sentiment_target', 'text_clean', 'text_stemmed', 'text_lemmatized']

Missing Values:
product_id               0
category                 0
subcategory1             0
subcategory2             0
location                 0
channel                  0
customer_age             0
review_title             0
review_text              0
rating                   0
recommend_flag           0
vader_negative           0
vader_neutral            0
vader_positive           0
vader_compound           0
vader_sentiment_score    0
vader_sentiment_label    0
age_cluster_raw          0
age_group                0
sentiment_target         

,product_id,category,subcategory1,subcategory2,location,channel,customer_age,review_title,review_text,rating,...,vader_positive,vader_compound,vader_sentiment_score,vader_sentiment_label,age_cluster_raw,age_group,sentiment_target,text_clean,text_stemmed,text_lemmatized
0,767,Initmates,Intimate,Intimates,Mumbai,Mobile,33,Not Provided,Absolutely wonderful - silky and sexy and comfortable,4,...,0.681,0.8932,0.94660,Positive,0,Age Group 2,Positive,absolutely wonderful silky and sexy and comfortable,absolut wonder silki and sexi and comfort,absolutely wonderful silky and sexy and comfortable
1,1080,General,Dresses,Dresses,Bangalore,Mobile,34,Not Provided,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",5,...,0.298,0.9729,0.98645,Positive,0,Age Group 2,Positive,love this dress it s sooo pretty i happened to find it in a store and i m glad i did bc i never would have ordered it online bc it s petite i bought a petite and am i love the length on me hits just a little below the knee would definitely be a true midi on someone who is truly petite,love this dress it s sooo pretti i happen to find it in a store and i m glad i did bc i never would have order it onlin bc it s petit i bought a petit and am i love the length on me hit just a littl below the knee would definit be a true midi on someon who is truli petit,love this dress it s sooo pretty i happened to find it in a store and i m glad i did bc i never would have ordered it online bc it s petite i bought a petite and am i love the length on me hit just a little below the knee would definitely be a true midi on someone who is truly petite
2,1077,General,Dresses,Dresses,Gurgaon,Mobile,60,Some major design flaws,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c",3,...,0.144,0.9208,0.96040,Positive,1,Age Group 4,Neutral,i had such high hopes for this dress and really wanted it to work for me i initially ordered the petite small my usual size but i found this to be outrageously small so small in fact that i could not zip it up i reordered it in petite medium which was just ok overall the top half was comfortable and fit nicely but the bottom half had a very tight under layer and several somewhat cheap net over layers imo a major design flaw was the net over layer sewn directly into the zipper it c,i had such high hope for this dress and realli want it to work for me i initi order the petit small my usual size but i found this to be outrag small so small in fact that i could not zip it up i reorder it in petit medium which was just ok overal the top half was comfort and fit nice but the bottom half had a veri tight under layer and sever somewhat cheap net over layer imo a major design flaw was the net over layer sewn direct into the zipper it c,i had such high hope for this dress and really wanted it to work for me i initially ordered the petite small my usual size but i found this to be outrageously small so small in fact that i could not zip it up i reordered it in petite medium which wa just ok overall the top half wa comfortable and fit nicely but the bottom half had a very tight under layer and several somewhat cheap net over layer imo a major design flaw wa the net over layer sewn directly into the zipper it c
3,1049,General Petite,Bottoms,Pants,Chennai,Web,50,My favorite buy!,"I love, love,

In [2]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical Columns:", categorical_cols)

print("\nUnique Categories in Each Column:")
for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")

Categorical Columns: ['category', 'subcategory1', 'subcategory2', 'location', 'channel', 'review_title', 'review_text', 'vader_sentiment_label', 'age_group', 'sentiment_target', 'text_clean', 'text_stemmed', 'text_lemmatized']

Unique Categories in Each Column:
category: 3 unique values
subcategory1: 6 unique values
subcategory2: 20 unique values
location: 4 unique values
channel: 2 unique values
review_title: 13994 unique values
review_text: 22635 unique values
vader_sentiment_label: 3 unique values
age_group: 4 unique values
sentiment_target: 3 unique values
text_clean: 22634 unique values
text_stemmed: 22634 unique values
text_lemmatized: 22634 unique values


In [5]:
# Analyze categorical cardinality

categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

cardinality_report = pd.DataFrame({
    "Column": categorical_cols,
    "Unique_Values": [
        df[col].nunique() for col in categorical_cols
    ],
    "Unique_Percentage": [
        round(df[col].nunique() / len(df) * 100, 2)
        for col in categorical_cols
    ]
})

cardinality_report = cardinality_report.sort_values(
    "Unique_Values"
)

display(cardinality_report)

,Column,Unique_Values,Unique_Percentage
4,channel,2,0.01
0,category,3,0.01
7,vader_sentiment_label,3,0.01
9,sentiment_target,3,0.01
8,age_group,4,0.02
3,location,4,0.02
1,subcategory1,6,0.03
2,subcategory2,20,0.09
5,review_title,13994,59.59
12,text_lemmatized,22634,96.38


Categorical Encoding and Dimensionality Reduction

In [3]:


# Select meaningful categorical columns
low_cardinality_cols = [
    "category",
    "subcategory1",
    "location",
    "channel",
    "vader_sentiment_label",
    "age_group"
]

high_cardinality_cols = [
    "subcategory2",
    "review_title"
]

print("Dataset Shape:", df.shape)
print("Low-cardinality columns:", low_cardinality_cols)
print("High-cardinality columns:", high_cardinality_cols)

Dataset Shape: (23483, 23)
Low-cardinality columns: ['category', 'subcategory1', 'location', 'channel', 'vader_sentiment_label', 'age_group']
High-cardinality columns: ['subcategory2', 'review_title']


One-Hot Encoding

In [4]:
one_hot_df = pd.get_dummies(df[low_cardinality_cols], columns=low_cardinality_cols, drop_first=False, dtype=int)

print("Original categorical shape:", df[low_cardinality_cols].shape)
print("One-hot encoded shape:", one_hot_df.shape)

display(one_hot_df.head())

Original categorical shape: (23483, 6)
One-hot encoded shape: (23483, 22)


,category_General,category_General Petite,category_Initmates,subcategory1_Bottoms,subcategory1_Dresses,subcategory1_Intimate,subcategory1_Jackets,subcategory1_Tops,subcategory1_Trend,location_Bangalore,...,location_Mumbai,channel_Mobile,channel_Web,vader_sentiment_label_Negative,vader_sentiment_label_Neutral,vader_sentiment_label_Positive,age_group_Age Group 1,age_group_Age Group 2,age_group_Age Group 3,age_group_Age Group 4
0,0,0,1,0,0,1,0,0,0,0,...,1,1,0,0,0,1,0,1,0,0
1,1,0,0,0,1,0,0,0,0,1,...,0,1,0,0,0,1,0,1,0,0
2,1,0,0,0,1,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,1
3,0,1,0,1,0,0,0,0,0,0,...,0,0,1,0,0,1,0,0,1,0
4,1,0,0,0,0,0,0,1,0,1,...,0,0,1,0,0,1,0,0,1,0


Frequency-Based Binning

In [6]:
def frequency_binning(series, min_frequency=0.001):
    series = series.fillna("Missing").astype(str)

    frequencies = series.value_counts(normalize=True)

    common_categories = frequencies[frequencies >= min_frequency].index

    return series.where(series.isin(common_categories), "Other")


df["review_title_binned"] = frequency_binning(df["review_title"],min_frequency=0.001)

print("Original unique titles:", df["review_title"].nunique())
print("After binning:", df["review_title_binned"].nunique())

display(df["review_title_binned"].value_counts().head(10).to_frame("Frequency"))

Original unique titles: 13994
After binning: 46


,Frequency
review_title_binned,
Other,17820
Not Provided,3807
Love it!,136
Beautiful,95
Love,88
Love!,84
Beautiful!,72
Beautiful dress,60
Love it,59


Target Encoding

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import TargetEncoder

In [8]:
x = df[['subcategory2']].copy()
y = df['rating'].copy()

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)

target_encoder = TargetEncoder(target_type='continuous', smooth='auto', random_state=42)

x_train_encoded = target_encoder.fit_transform(x_train, y_train)
x_test_encoded = target_encoder.transform(x_test)

target_encoded_train = pd.DataFrame(x_train_encoded, columns=["subcategory2_target_encoded"],index=x_train.index)

target_encoded_test = pd.DataFrame(x_test_encoded, columns=["subcategory2_target_encoded"],index=x_test.index)

print("Training encoded shape:", target_encoded_train.shape)
print("Testing encoded shape:", target_encoded_test.shape)

display(target_encoded_train.head())

Training encoded shape: (18786, 1)
Testing encoded shape: (4697, 1)


,subcategory2_target_encoded
17984,4.159326
21328,4.148961
8113,4.156547
23033,4.155833
316,4.148961


In [10]:
%pip install category-encoders

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from category_encoders import TargetEncoder

target_encoder = TargetEncoder(cols=["subcategory2"],smoothing=10)

X_train_encoded = target_encoder.fit_transform(x_train, y_train)
X_test_encoded = target_encoder.transform(x_test)

Dimensionality Reduction using Truncated SVD

In [12]:
from sklearn.decomposition import TruncatedSVD

if one_hot_df.shape[1] >= 3:
    svd = TruncatedSVD(n_components=2, random_state=42)

    reduced_array = svd.fit_transform(one_hot_df)

    reduced_df = pd.DataFrame(reduced_array, columns=["SVD_Component_1", "SVD_Component_2"], index=df.index)

    print("Before dimensionality reduction:", one_hot_df.shape)
    print("After dimensionality reduction:", reduced_df.shape)

Before dimensionality reduction: (23483, 22)
After dimensionality reduction: (23483, 2)


Compare the techniques

In [13]:
comparison_df = pd.DataFrame({
    "Technique": [
        "One-Hot Encoding",
        "Frequency-Based Binning",
        "Target Encoding",
        "Truncated SVD"
    ],
    "Purpose": [
        "Convert categories into binary columns",
        "Group rare categories into Other",
        "Encode categories using target statistics",
        "Reduce the number of dimensions"
    ],
    "Benefit": [
        "Easy to interpret",
        "Controls category growth",
        "Compact representation for categories",
        "Reduces feature dimensions"
    ],
    "Limitation": [
        "May increase the number of features",
        "May merge distinct categories",
        "Requires leakage prevention",
        "Reduced components are less interpretable"
    ]
})
display(comparison_df)

,Technique,Purpose,Benefit,Limitation
0,One-Hot Encoding,Convert categories into binary columns,Easy to interpret,May increase the number of features
1,Frequency-Based Binning,Group rare categories into Other,Controls category growth,May merge distinct categories
2,Target Encoding,Encode categories using target statistics,Compact representation for categories,Requires leakage prevention
3,Truncated SVD,Reduce the number of dimensions,Reduces feature dimensions,Reduced components are less interpretable


Save Task 21 outputs

In [14]:
# Save one-hot encoded data
one_hot_df.to_csv("task21_one_hot_encoded.csv",index=False)

# Save frequency-binned titles
df[["review_title", "review_title_binned"]].to_csv("task21_frequency_binned.csv",index=False)

# Save target-encoded train and test data
target_encoded_train.to_csv("task21_target_encoded_train.csv",index=False)

target_encoded_test.to_csv("task21_target_encoded_test.csv",index=False)

# Save dimensionality-reduced data
reduced_df.to_csv("task21_svd_reduced.csv",index=False)

print("Task 21 outputs saved successfully!")

Task 21 outputs saved successfully!


In [15]:
# Create final Task 21 dataset
task21_df = df.copy()

# Add one-hot encoded features
task21_df = pd.concat([task21_df, one_hot_df],axis=1)

# Add frequency-binned title
task21_df["review_title_binned"] = df["review_title_binned"]

# Save final dataset
task21_df.to_csv("task21_final_dataset.csv",index=False)

print("Task 21 final dataset saved successfully!")
print("Dataset shape:", task21_df.shape)
print("Saved file: task21_final_dataset.csv")

Task 21 final dataset saved successfully!
Dataset shape: (23483, 46)
Saved file: task21_final_dataset.csv


### Conclusion – Task 21

In this task, categorical encoding and dimensionality reduction techniques were applied to the Women’s Clothing Reviews dataset containing 23,483 records and 23 columns. The dataset had no missing values. Cardinality analysis showed that `category`, `subcategory1`, `location`, `channel`, `vader_sentiment_label`, and `age_group` were low-cardinality features, whereas `review_title` had 13,994 unique values and the review-text-related columns had more than 22,600 unique values. One-Hot Encoding converted six categorical columns into 22 binary features. Frequency-Based Binning reduced the unique review titles from 13,994 to 46 categories; however, 17,820 records were assigned to “Other,” indicating that many titles occurred infrequently. The 3,807 “Not Provided” titles remained a separate category. Target Encoding represented `subcategory2` using average ratings calculated from training data, producing one numerical feature. The encoded training and testing sets contained 18,786 and 4,697 records, respectively. Truncated SVD reduced the 22 one-hot encoded features to two components. Overall, these techniques demonstrated how feature expansion can be controlled. However, the SVD explained-variance results and the effect of each encoding technique on rating prediction performance should be evaluated before deciding which representation is most suitable for the final model.